# ZAIC DataMate — preflight và đối chiếu phiên bản

Notebook mới chỉ chứa code và kiểm thử giả lập. Mặc định CPU preflight; không tải model, không huấn luyện, không nộp bài.

GPU smoke 9B cần đủ base và hash nguồn. Train cần bằng chứng được phép đưa dữ liệu/SFT lên Kaggle. Private không thay bằng chứng.


In [ ]:
CONFIG = {'ACTION': 'preflight', 'BASE_INPUT': '/kaggle/input/REVIEW_BASE/base', 'SFT_INPUT': '/kaggle/input/REVIEW_APPROVED_SFT/data', 'GPU_RUN_CONFIRMED': False, 'COMPETITION_CLOUD_USE_ALLOWED': False, 'CLOUD_PERMISSION_EVIDENCE': '', 'TRAIN_FRESH_ADAPTER': False, 'BASE_FILE_HASHES': {}}


In [ ]:
# Code + synthetic tests only. No competition data or weights embedded.
from pathlib import Path
from datetime import datetime, timezone
import base64, hashlib, io, json, stat, sys, zipfile
PAYLOAD = "..."
EXPECTED_SHA256 = "5a32f9c2bdf8b8cdda143fb18062549780a75ce2499c3889d403dd77b905e94e"
raw = base64.b64decode(PAYLOAD)
assert hashlib.sha256(raw).hexdigest() == EXPECTED_SHA256, 'Code archive checksum mismatch'
source = Path('/kaggle/temp/zaic_code') / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
source.mkdir(parents=True, exist_ok=False)
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    assert sum(i.file_size for i in z.infolist()) < 25 * 1024 * 1024
    for i in z.infolist():
        assert (source / i.filename).resolve().is_relative_to(source.resolve())
        assert not stat.S_ISLNK(i.external_attr >> 16)
    z.extractall(source)
sys.path.insert(0, str(source / 'gui'))
from versioned_runner import run
RESULT = run(CONFIG, source)


## Lưu và đối chiếu
Sau completed, lưu phiên bản với output hiện tại (tránh chạy lại train). Đợi version hoàn tất. Tải ZIP của đúng run; đối chiếu ARTIFACT_MANIFEST SHA256 của bản đã lưu với máy. Chỉ sau đó cập nhật LATEST_SYNCED. BEST_MODEL chỉ chọn bằng đánh giá TRAIN.

Base và SFT giữ ở input, scratch ở /kaggle/temp; output chỉ có checkpoint/adapter và báo cáo. Runner này chưa hỗ trợ resume: giữ checkpoint đầy đủ để triển khai resume riêng sau khi kiểm fingerprint.
